# Lab: Automatic Speech Recognition on PARLO
**Duration:** about 90 minutes.

In the PStA you will use **text** as the second modality.
In a real application nobody writes CHAT transcripts by hand: the text comes from an ASR system.
Today you find out what that means:
- set up a local ASR system (whisper.cpp, Whisper or Parakeet) and play around with it,
- transcribe PARLO recordings and evaluate them against the CHAT references with **WER**, **CER** and **BERTScore**,
- compare the error rates of the three groups HC / MCI / AD,
- discuss how the exactness of the transcript influences your classification task.

Coding is **not** the point of this lab.
The code cells are skeletons with hints and `TODO`s; feel free to let your coding agent fill them in.
Spend your time on choosing the setup, looking at the transcripts and answering the questions.

> **Legal reminder:** the PARLO data must not leave your machine.
> Use **local** ASR only: no OpenAI API, no online transcription services, no cloud notebooks.
> Do not upload audio, transcripts or notebook outputs to GitHub or other online services.

# Part 0: Setup (10 min)
Choose **one** system (you can add a second one later, see Part 4).
All options run locally, and all of them can be used from this notebook, so you do not need the shell unless you want to build whisper.cpp yourself.

| Option | What | Install | Pros | Cons |
|---|---|---|---|---|
| A | [whisper.cpp](https://github.com/ggml-org/whisper.cpp) CLI (`whisper-cli`) | brew / build / release zip | fast on CPU and Apple Silicon, quantized models, VAD, prompt | needs an install outside Python |
| B | whisper.cpp from Python ([`pywhispercpp`](https://github.com/absadiki/pywhispercpp)) | `pip install pywhispercpp` | same engine as A, no shell | wrapper, options named slightly differently |
| C | Whisper via `transformers` | already in the course environment | no extra install, same API as Lab 03_01 | slow on CPU, chunking needed for audio > 30 s |
| D | Parakeet TDT via `transformers` | already in the course environment | different architecture (transducer instead of encoder-decoder), multilingual incl. German | large download (about 2.5 GB), no prompt |

**Option A: installing whisper.cpp**
- macOS: `brew install whisper-cpp` (installs `whisper-cli`)
- Linux / macOS from source:
  ```bash
  git clone https://github.com/ggml-org/whisper.cpp
  cd whisper.cpp
  cmake -B build && cmake --build build -j --config Release   # binary: build/bin/whisper-cli
  ```
- Windows: download a prebuilt release zip from the [GitHub releases](https://github.com/ggml-org/whisper.cpp/releases), build with CMake and Visual Studio, or use WSL and follow the Linux steps
- Models: `sh ./models/download-ggml-model.sh base` (or `large-v3-turbo`), or download a `ggml-*.bin` file from the Hugging Face repo [`ggerganov/whisper.cpp`](https://huggingface.co/ggerganov/whisper.cpp) (the next cell does that for you)

**Hints**
- Start with a **small model** (`base`) on a few recordings, and switch to a larger one (`large-v3-turbo`) once everything works.
- Transcribing 200 recordings takes a while: store the hypotheses in a file (as with the features in the previous labs) and never transcribe twice.
- The recordings are German, but whisper.cpp defaults to **English** (`-l en`)!

In [ ]:
# run if needed (the course uv environment already contains most of it):
#!pip install jiwer bert-score pywhispercpp

In [ ]:
import os
import re
import subprocess
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
import torch
from tqdm.auto import tqdm
from IPython.display import display

import jiwer
from parlo import Corpus

DEVICE = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
print('device:', DEVICE)

# transcripts are cached outside the repo, as in the previous labs (override with SLP_CACHE)
CACHE_DIR = Path(os.environ.get('SLP_CACHE', Path.home() / '.cache' / 'slp_labs'))

PARLO_ROOT = Path(os.environ.get('PARLO_ROOT', '~/data/PARLO')).expanduser()
if not (PARLO_ROOT / '0metadata.csv').exists():
    raise FileNotFoundError(f'No PARLO corpus at {PARLO_ROOT}: set PARLO_ROOT or edit the fallback path above')

corpus = Corpus(PARLO_ROOT)
TASK = 'pictdes'

## The four systems
Each option is wrapped in a function `transcribe(path) -> str` that takes the path of a WAV file and returns the raw transcript.
Run **only the cell of your option**.

In [ ]:
# Option A: whisper.cpp CLI
from huggingface_hub import hf_hub_download

WHISPER_CLI = 'whisper-cli'   # or the full path, e.g. 'whisper.cpp/build/bin/whisper-cli' (Windows: whisper-cli.exe)
GGML_MODEL = hf_hub_download('ggerganov/whisper.cpp', 'ggml-base.bin')   # later: 'ggml-large-v3-turbo.bin'


def transcribe(path, prompt=None):
    cmd = [WHISPER_CLI, '-m', GGML_MODEL, '-f', str(path),
           '-l', 'de',     # German! the default is English
           '-nt', '-np']   # no timestamps, no log output: stdout is just the text
    if prompt:
        cmd += ['--prompt', prompt]
    # more options to try: '-bs' (beam size), '--vad' with '-vm' (VAD model), '-l auto', see `whisper-cli -h`
    out = subprocess.run(cmd, capture_output=True, text=True, check=True)
    return ' '.join(out.stdout.split())

In [ ]:
# Option B: whisper.cpp from Python (pip install pywhispercpp)
from pywhispercpp.model import Model

whisper_cpp = Model('base', print_progress=False, print_realtime=False)   # downloads the ggml model on first use


def transcribe(path, prompt=None):
    # TODO: German, and optionally the initial prompt; check the pywhispercpp README for the parameter names
    segments = whisper_cpp.transcribe(str(path), language='de')
    return ' '.join(s.text.strip() for s in segments)

In [ ]:
# Option C: Whisper via transformers
from transformers import pipeline

pipe = pipeline('automatic-speech-recognition', model='openai/whisper-large-v3-turbo', device=DEVICE)


def transcribe(path, prompt=None):
    # the pipeline decodes and resamples the file itself (needs ffmpeg);
    # Whisper sees 30 s windows, so long recordings need return_timestamps=True (or chunk_length_s=30)
    out = pipe(str(path), return_timestamps=True, generate_kwargs={'language': 'german', 'task': 'transcribe'})
    return out['text'].strip()

In [ ]:
# Option D: Parakeet TDT via transformers
from transformers import pipeline

pipe = pipeline('automatic-speech-recognition', model='nvidia/parakeet-tdt-0.6b-v3', device=DEVICE)


def transcribe(path, prompt=None):
    # no language argument: the model detects the language itself; no prompt either
    return pipe(str(path))['text'].strip()

# Part 1: Playing Around (15 min)
`corpus.asr(...)` returns the recordings together with their reference transcripts, more on that in Part 2.
Here we only need a few audio paths.

Transcribe two or three recordings, ideally from different tasks (`pictdes`, `recall`, `animal`) and from different groups, and compare them with the reference **by eye**.
Then try some of the knobs of your system:
- model size (`base` vs. `large-v3-turbo`),
- language `de` vs. auto-detection,
- an initial prompt (options A to C), e.g. `prompt='Äh, also, ähm, ja, ich, ich weiß nicht.'`,
- beam size, VAD (option A).

In [ ]:
examples = (corpus.filter(task=['pictdes', 'recall', 'animal']).asr('recording')
            .group_by('task', 'diagnosis').first()          # one recording per task and group
            .sort('task', 'diagnosis').collect())


def wav_path(row):
    p = Path(row['audio_path'])
    return p if p.is_absolute() else PARLO_ROOT / p


row = examples.row(0, named=True)        # TODO: try other rows
print(row['task'], row['diagnosis'])
print('REF:', row['reference_verbatim'])
print('HYP:', transcribe(wav_path(row)))

**Questions**
1. What happens to **fillers** ("äh", "ähm", "mhm"), **repetitions** ("ein ein Stück weiter"), **self-corrections** and **word fragments** ("Ge Gerät")?
   Which system keeps them, which one "tidies up" the speech?
2. Is the **interviewer** in the hypothesis? In the reference?
3. What changes with the model size, the language setting and the prompt?
   Does a prompt full of fillers make Whisper more verbatim?
4. Do you see **hallucinations** (text that was never spoken, e.g. at the end of a recording or in long pauses)?

# Part 2: Transcribing a Task (10 min)
`corpus.filter(task=TASK).asr(level)` prepares an ASR evaluation:
- `level='recording'`: one row per recording with `audio_path`, the CHAT transcript of **all speakers** as `reference_verbatim`, and a normalized `reference` (lowercase, no punctuation)
- `level='utterance'`: one row per **utterance** with `start_ms` / `end_ms`; you could cut out the participant's utterances and transcribe only those

Only recordings with a valid reference are returned (no placeholders such as `xxx`).

**Hint:** start with `.head(10)`; when everything works, run the full task and save the hypotheses.

In [ ]:
data = corpus.filter(task=TASK).asr('recording').collect()
print(data.shape)
data.group_by('diagnosis').len().sort('diagnosis')

In [ ]:
HYP_FILE = CACHE_DIR / f'asr_{TASK}_SYSTEM.parquet'   # TODO: put your system / model name instead of SYSTEM

if HYP_FILE.exists():
    hyps = pl.read_parquet(HYP_FILE)
else:
    rows = data.head(10).iter_rows(named=True)       # TODO: remove .head(10) for the full run
    hyps = pl.DataFrame([{'recording_id': r['recording_id'], 'hypothesis_verbatim': transcribe(wav_path(r))}
                         for r in tqdm(rows)])
    # TODO: save only the full run, e.g. HYP_FILE.parent.mkdir(parents=True, exist_ok=True); hyps.write_parquet(HYP_FILE)

results = data.select('recording_id', 'participant_id', 'diagnosis', 'reference_verbatim', 'reference').join(hyps, on='recording_id')
results.head()

# Part 3: Metrics (20 min)

## Normalization
WER compares **strings of words**: "Rucksack," and "rucksack" are two different words.
Before scoring, reference and hypothesis must be brought to the same form.
A simple normalization looks like this:

```python
import re
import unicodedata


def normalize(text: str) -> str:
    """Unicode-normalize, lowercase, strip punctuation, collapse whitespace."""
    text = unicodedata.normalize("NFKC", text)
    text = text.lower()
    text = "".join(ch for ch in text if not unicodedata.category(ch).startswith("P"))
    text = re.sub(r"\s+", " ", text).strip()
    return text
```

The `reference` column was normalized in a very similar way by the loader.
Apply your normalization to the hypotheses, e.g. as a new column:

```python
results = results.with_columns(
    pl.col('hypothesis_verbatim').map_elements(normalize, return_dtype=pl.String).alias('hypothesis'))
```

**Questions**
1. Why must the hypothesis be normalized with **the same** function as the reference?
   Compute the WER once with the raw hypothesis and once with the normalized one.
2. What else would a "fair" normalization have to handle? Think of numbers ("drei" vs. "3"), compounds ("Rollkoffer" vs. "Roll Koffer"), spelling variants, CHAT conventions in the reference.
3. Should **fillers** be removed from the reference before scoring?
   That depends on what you want to measure: the quality of the ASR for *content*, or the faithfulness for *everything that was said*.
   Which of the two matters for your PStA?

In [ ]:
# TODO: normalization of the hypotheses (and maybe a stricter version for both sides)

## WER, CER and BERTScore
- **WER** = (S + D + I) / N on words, **CER** the same on characters.
- **BERTScore** matches the contextual embeddings of reference and hypothesis tokens (precision, recall, F1).
  It rewards a hypothesis that *means* the same, even with different words.

With `jiwer` and `bert_score` this is only a few lines.
Note that both take **lists** of references and hypotheses:

```python
import jiwer
from bert_score import BERTScorer

scorer = BERTScorer(lang='de', rescale_with_baseline=True)   # load the model only once!


def evaluate(refs, hyps):
    wer = jiwer.wer(refs, hyps)            # corpus-level: all errors pooled, / all reference words
    cer = jiwer.cer(refs, hyps)
    _, _, f1 = scorer.score(hyps, refs)    # one F1 per pair (candidates first, then references)
    return {'wer': wer, 'cer': cer, 'bertscore_f1': f1.mean().item()}
```

To see *what* goes wrong, look at the alignment and the error counts:

```python
out = jiwer.process_words(ref, hyp)
print(out.substitutions, out.deletions, out.insertions)
print(jiwer.visualize_alignment(out))
```

**Warm-up:** before you score PARLO, try it on this excerpt of a story recall (reference and two ASR systems):

```python
ref = ['Schmiss diesen Rucksack über das Drehkreuz und rollte dann mit dem Rollkoffer DURCH das Drehkreuz Und der Pförtner der nebenan saß lächelte und grinste']
system_a = ['schmiss diesen Ucksack über das Drehkreuz und rollte dann mit dem Rollkoffer durch das Drehkreuz. Und der Pförtner, der nebenan saß, lächelte und grinste']
system_b = ['Schmiss diesen Rucksack über das Drehkreuz und rödel mit dem Röckhofer durch das Drehkreuz. Und der Pörtner, der nebenan saß lächelte und grünste']
```

**Questions**
1. Warm-up: which system is better according to WER, CER and BERTScore, with and without normalization? Do the metrics agree?
2. What happens to BERTScore with `lang='en'` for German text? Why?
3. `jiwer.wer` on lists is the corpus-level WER, not the mean of the per-recording WERs.
   When do they differ? Which one do you want to compare groups whose recordings have different lengths?
4. Can the WER be larger than 100 %? When?
5. On PARLO: find a hypothesis with a high WER but a high BERTScore, and one where it is the other way round. What kind of errors are these?
6. "Ucksack" vs. "Röckhofer": which kind of error hurts a **text classifier** most, and does that depend on the features you use?

In [ ]:
# TODO: warm-up example

In [ ]:
# TODO: evaluate your system on PARLO (normalized reference vs. normalized hypothesis)

In [ ]:
# TODO: look at the alignment of the worst recordings

# Part 4: Error Rates per Group (20 min)
Now the interesting part: is the ASR equally good for everybody?

For **one system and one task** compute:
1. WER, CER and BERTScore F1 **per recording** (add them as columns to `results`),
2. the corpus-level metrics **per diagnosis group** (HC / MCI / AD),
3. a boxplot of the per-recording WER per group.

**Hints**
- `results.group_by('diagnosis')` and your `evaluate` function, or `map_elements` / a loop for the per-recording values.
- With about 60 recordings per group, a difference of a few percentage points may be noise: look at the spread in the boxplot (optionally `scipy.stats.kruskal`).

**Optional**
- a second system (e.g. whisper.cpp vs. Parakeet) or a larger model on the same task,
- a second task (`recall`, `animal`, `reading`): where is ASR easiest, where hardest, and why?

In [ ]:
# TODO: per-recording metrics, per-group table, boxplot

# Part 5: What Does This Mean for Your PStA? (15 min)
Discuss in your team and write down short answers.

1. If the WER is higher for AD than for HC: is that a problem of the ASR, or **information** about the speaker?
   Could the WER (or the ASR confidence) itself be a feature?
2. Fillers, repetitions, fragments and pauses are known markers of cognitive decline.
   What happens to them in your ASR output?
   Which of your planned text features survive ASR, which ones disappear?
3. You train a text classifier on the CHAT transcripts and apply it to ASR transcripts (or the other way round).
   What happens? How would you avoid this train/test mismatch?
4. Which metric do you expect to predict the downstream **classification accuracy** better: WER (exact words) or BERTScore (meaning)?
   Does the answer depend on the features (counts of fillers and repetitions vs. sentence embeddings vs. information units of the picture description)?
5. The recording-level reference contains the **interviewer**, and so may the ASR output.
   How could the interviewer's speech leak into your classifier?
6. For the PStA: reference transcripts, ASR transcripts, or both?
   What would be realistic for an application in a memory clinic?